# 🚗 Car Market Trends Analysis & Price Prediction
### **Internship Project: Exploratory Data Analysis & Machine Learning Valuation Engine**
---
**Author:** Data Analytics Intern  
**Dataset:** Car Dekho Used Vehicle Dataset  
**Objectives:**
1. Perform comprehensive exploratory data analysis (EDA) to discover key drivers of used car valuation.
2. Analyze market depreciation rates across vehicle age, fuel types, transmission mechanisms, and seller channels.
3. Engineer features to optimize predictive performance.
4. Train and benchmark multiple Machine Learning regression models (Linear Regression, Ridge, Random Forest, Gradient Boosting).
5. Extract feature importances and formulate data-driven business insights for dealerships and buyers.

## 1. Environment Setup & Library Imports

In [ ]:
# Google Colab File Upload (Optional: If running in Google Colab and dataset is not yet uploaded)
import os
csv_filename = '1776311302-P3-Car Market Trends Analysis with Car Dekho Data.csv'
if not os.path.exists(csv_filename):
    try:
        from google.colab import files
        print('Please upload the Car Dekho CSV dataset:')
        uploaded = files.upload()
    except ImportError:
        pass


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Visual styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('deep')
warnings.filterwarnings('ignore')
%matplotlib inline

## 2. Data Ingestion & Initial Inspection

In [ ]:
# Load Car Dekho Dataset
df = pd.read_csv('1776311302-P3-Car Market Trends Analysis with Car Dekho Data.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head(10)

In [ ]:
# Dataset structural summary
df.info()

In [ ]:
# Statistical overview of numeric variables
df.describe().T

## 3. Data Cleaning & Integrity Verification

In [ ]:
# Check for null values across features
missing_vals = df.isnull().sum()
print("Missing values per column:")
print(missing_vals)

# Check for duplicate entries
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicate_count}")
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Cleaned dataset shape: {df.shape}")

## 4. Feature Engineering
We derive domain-relevant features:
- **`Car_Age`**: Age of the vehicle in years (`Current_Year - Manufacturing_Year`).
- **`Depreciation_Value`**: Absolute depreciation (`Present_Price - Selling_Price`).
- **`Depreciation_Percent`**: Percentage depreciation relative to original showroom price.
- **`Price_Retention_Ratio`**: Proportion of original value retained.

In [ ]:
current_year = 2026
df['Car_Age'] = current_year - df['Year']
df['Depreciation_Value'] = df['Present_Price'] - df['Selling_Price']
df['Depreciation_Percent'] = ((df['Depreciation_Value'] / df['Present_Price']) * 100).round(2)
df['Price_Retention_Ratio'] = (df['Selling_Price'] / df['Present_Price']).round(4)

df[['Car_Name', 'Year', 'Car_Age', 'Present_Price', 'Selling_Price', 'Depreciation_Percent']].head()

## 5. Exploratory Data Analysis (EDA) & Market Trend Discovery

In [ ]:
# 5.1 Distribution of Selling Price and Present Price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Selling_Price'], kde=True, ax=axes[0], color='#2563eb', bins=25)
axes[0].set_title('Distribution of Selling Price (Lakhs INR)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Selling Price (in Lakhs)')

sns.histplot(df['Present_Price'], kde=True, ax=axes[1], color='#059669', bins=25)
axes[1].set_title('Distribution of Present Showroom Price (Lakhs INR)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Present Price (in Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Price Depreciation Analysis by Vehicle Age
plt.figure(figsize=(10, 6))
sns.lineplot(x='Car_Age', y='Selling_Price', data=df, marker='o', color='#dc2626', linewidth=2.5, errorbar=('ci', 95))
plt.title('Vehicle Selling Price Decay vs. Car Age (Years)', fontsize=14, fontweight='bold')
plt.xlabel('Car Age (Years)', fontsize=12)
plt.ylabel('Average Selling Price (Lakhs)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 5.3 Valuation Breakdown by Fuel Type, Transmission, and Seller Type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x='Fuel_Type', y='Selling_Price', data=df, ax=axes[0], palette='Blues_d', ci=None)
axes[0].set_title('Avg Selling Price by Fuel Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (Lakhs)')

sns.barplot(x='Transmission', y='Selling_Price', data=df, ax=axes[1], palette='Greens_d', ci=None)
axes[1].set_title('Avg Selling Price by Transmission', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Price (Lakhs)')

sns.barplot(x='Seller_Type', y='Selling_Price', data=df, ax=axes[2], palette='Purples_d', ci=None)
axes[2].set_title('Avg Selling Price by Seller Channel', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Price (Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Correlation Matrix & Feature Interaction
plt.figure(figsize=(9, 7))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 5.5 Kms Driven vs Selling Price with Fuel Type Segmentation
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Kms_Driven', y='Selling_Price', hue='Fuel_Type', size='Present_Price', data=df, sizes=(40, 250), alpha=0.8, palette='Set1')
plt.title('Selling Price vs Mileage (Kms Driven)', fontsize=14, fontweight='bold')
plt.xlabel('Kilometers Driven', fontsize=12)
plt.ylabel('Selling Price (Lakhs)', fontsize=12)
plt.show()

## 6. Machine Learning Regression Modeling & Benchmarking
We train four distinct regression algorithms:
1. **Linear Regression (Baseline)**
2. **Ridge Regression (L2 Regularized)**
3. **Random Forest Regressor (Ensemble Bagging)**
4. **Gradient Boosting Regressor (Ensemble Boosting)**

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Feature definition
feature_cols = ['Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'Car_Age']
X = df[feature_cols]
y = df['Selling_Price']

cat_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']
num_cols = ['Present_Price', 'Kms_Driven', 'Owner', 'Car_Age']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

# Train-test partition (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

benchmark_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    
    preds = pipe.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    benchmark_results.append({
        'Model': name,
        'R2 Score (Test)': round(r2, 4),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4)
    })

results_df = pd.DataFrame(benchmark_results).sort_values(by='R2 Score (Test)', ascending=False)
print("Model Benchmarking Summary:")
results_df

In [ ]:
# 6.2 Visual Comparison of Actual vs Predicted Prices
best_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))])
best_pipe.fit(X_train, y_train)
y_pred_best = best_pipe.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, color='#2563eb', alpha=0.8, edgecolor='k', s=70)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Fit ($y=x$)')
plt.title('Gradient Boosting: Actual vs Predicted Selling Prices', fontsize=14, fontweight='bold')
plt.xlabel('Actual Selling Price (Lakhs)', fontsize=12)
plt.ylabel('Predicted Selling Price (Lakhs)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 7. Feature Importance Analysis

In [ ]:
rf_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_pipe.fit(X, y)

cat_trans = rf_pipe.named_steps['preprocessor'].named_transformers_['cat']
encoded_cat_names = list(cat_trans.get_feature_names_out(cat_cols))
all_features = num_cols + encoded_cat_names
importances = rf_pipe.named_steps['regressor'].feature_importances_

feat_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis')
plt.title('Random Forest Feature Importance Analysis', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance Score')
plt.show()

## 8. Strategic Business Insights & Recommendations

### 📈 Key Findings:
1. **Dominant Valuation Driver**: `Present_Price` (original showroom cost) accounts for >85% of price determination, followed by `Car_Age` and `Kms_Driven`.
2. **Fuel Impact**: Diesel vehicles hold a higher average resale price in absolute terms compared to petrol counterparts due to commercial demand and highway efficiency.
3. **Transmission Premium**: Automatic cars command an observable price premium over manual transmissions across comparable age brackets.
4. **Seller Channel**: Dealership sales average significantly higher valuations than direct individual seller listings due to certified warranties and refurbishment.

### 💡 Next Steps:
- Integrate this trained valuation model into a real-time interactive valuation web dashboard for buyers and automotive dealerships.